In [ ]:
from pathlib import Path
import re
import shutil

import pandas as pd


# ============================================================================
# 17. STRIP ICLE DOCUMENT-ID HEADER (one-time repair)
# ============================================================================
# Notebook 16 confirms that 2619/2619 (100%) of ICLE rows carry a literal
# document-ID header ("<ICLE-XX-YYY-####.#>") at position 0 of the 'text'
# column, that 0% of TOEFL11 rows have it, and that 0% of ICLE rows have it
# in 'masked_text' (masking incidentally destroys it). This is a data
# hygiene problem in the corpus build, not a property of the essays.
#
# Notebook 01 now strips this header at source, so a pipeline built from
# scratch does not need this notebook. It is kept for corpora that were
# built before that fix: it repairs combined_nli_masked.csv in place,
# stripping the header from 'text' only, after backing up the original file,
# and verifies that no contamination remains.
#
# AFTER RUNNING THIS: every notebook that uses full-text ICLE
# (03/04/05/07/08/09/10/11/12/13/15) must be re-run against the repaired
# file. Their code does not need to change, but any number produced before
# the repair was computed on contaminated text.

BASE_DIR = Path("..").resolve()
COMBINED_CSV_PATH = BASE_DIR / "data/processed/combined_nli_masked.csv"
BACKUP_CSV_PATH = BASE_DIR / "data/processed/combined_nli_masked_PRE_HEADER_FIX_BACKUP.csv"

HEADER_PATTERN = re.compile(r"^<ICLE-[A-Za-z]{2}-[A-Za-z0-9\-]+-\d+\.\d+>\s*")

if not COMBINED_CSV_PATH.exists():
    print(f"Skipping -- {COMBINED_CSV_PATH.name} not found.")
else:
    if BACKUP_CSV_PATH.exists():
        print(f"Backup already exists at {BACKUP_CSV_PATH.name} -- not overwriting it. "
              "Delete it manually first if you want a fresh backup of the current file.")
    else:
        shutil.copy(COMBINED_CSV_PATH, BACKUP_CSV_PATH)
        print(f"Backed up original file to: {BACKUP_CSV_PATH}")

    df = pd.read_csv(COMBINED_CSV_PATH)
    df["dataset"] = df["dataset"].astype(str).str.upper()
    df["text"] = df["text"].fillna("").astype(str)

    icle_mask = df["dataset"] == "ICLEV3"
    before_flagged = df.loc[icle_mask, "text"].str.contains(HEADER_PATTERN, regex=True).sum()
    print(f"\nBefore: {before_flagged} / {icle_mask.sum()} ICLE rows carry the header in 'text'.")

    df.loc[icle_mask, "text"] = df.loc[icle_mask, "text"].apply(lambda s: HEADER_PATTERN.sub("", s, count=1))

    after_flagged = df.loc[icle_mask, "text"].str.contains(HEADER_PATTERN, regex=True).sum()
    print(f"After:  {after_flagged} / {icle_mask.sum()} ICLE rows still carry the header in 'text'.")

    if after_flagged > 0:
        print(
            "\nWARNING: some rows still match after stripping -- the pattern may not be fully "
            "general (e.g. a different header format variant). Inspect these rows before trusting "
            "the cleaned file. NOT overwriting combined_nli_masked.csv until this is resolved."
        )
        residual = df.loc[icle_mask & df["text"].str.contains(HEADER_PATTERN, regex=True), ["doc_id", "text"]].head(10)
        for _, r in residual.iterrows():
            print(f"  {r['doc_id']}: {r['text'][:80]}")
    else:
        df.to_csv(COMBINED_CSV_PATH, index=False)
        print(f"\nSaved cleaned file to: {COMBINED_CSV_PATH}")
        print("\nSample of cleaned ICLE text (first 3 rows that had a header):")
        sample_ids = df.loc[icle_mask, "doc_id"].head(3)
        # The file has already been overwritten, so this shows the cleaned state.
        for doc_id in sample_ids:
            row_text = df.loc[df["doc_id"] == doc_id, "text"].values[0]
            print(f"  {doc_id}: {row_text[:100]}...")

        print(
            "\nNEXT STEPS: re-run, in order, every notebook that used full-text ICLE: "
            "03, 04, 05, 07, 08, 09, 10, 11, 12, 13, 15. Their code doesn't need to change -- "
            "they read from this same combined_nli_masked.csv path -- but every full-text-ICLE "
            "number they previously produced was computed on contaminated text and is now stale."
        )
